In [2]:
import sys
sys.path.append("..")

import os
import pandas as pd
import numpy as np
from src.utils import audit, check_disguised_missing

# Tự động nhận diện file .csv hoặc .xlsx trong data/raw/
raw_files = os.listdir("../data/raw")
print("Các file trong data/raw/:", raw_files)

csv_files = [f for f in raw_files if f.endswith(".csv")]
xlsx_files = [f for f in raw_files if f.endswith(".xlsx")]

if csv_files:
    df_raw = pd.read_csv(f"../data/raw/{csv_files[0]}", encoding="ISO-8859-1")
elif xlsx_files:
    path = f"../data/raw/{xlsx_files[0]}"
    df_raw = pd.concat([
        pd.read_excel(path, sheet_name="Year 2009-2010"),
        pd.read_excel(path, sheet_name="Year 2010-2011")
    ], ignore_index=True)

print(f"Kích thước dữ liệu gốc: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
display(audit(df_raw).sort_values("pct_missing", ascending=False))
display(check_disguised_missing(df_raw))

Các file trong data/raw/: ['online_retail_II.csv']
Kích thước dữ liệu gốc: 1,067,371 dòng, 8 cột


,dtype,n_missing,pct_missing,n_unique,sample
Customer ID,float64,243007,22.77,5942,13085.0
Description,str,4382,0.41,5698,15CM CHRISTMAS GLASS BALL 20 LIGHTS
StockCode,str,0,0.00,5305,85048
Invoice,str,0,0.00,53628,489434
Quantity,int64,0,0.00,1057,12
InvoiceDate,str,0,0.00,47635,2009-12-01 07:45:00
Price,float64,0,0.00,2807,6.95
Country,str,0,0.00,43,United Kingdom


,column,n_disguised,examples
0,Description,342,"[damaged, ?, check, ??]"
1,Country,756,[Unspecified]


In [3]:
df = df_raw.copy()
cleaning_log = []

def log_step(step, column, finding, action, rows_affected, rationale):
    cleaning_log.append({
        "Step": step,
        "Column": column,
        "Finding": finding,
        "Action": action,
        "Rows": int(rows_affected),
        "Why": rationale
    })

# 4.1 Chuẩn hóa tên cột (snake_case) và kiểu dữ liệu
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df["invoice_date"] = pd.to_datetime(df["invoicedate"])
df = df.drop(columns=["invoicedate"])

for col in ["invoice", "stockcode", "description", "country"]:
    df[col] = df[col].astype(str).str.strip()

log_step(
    "4.1", "all_columns",
    "Tên cột có khoảng trắng (Customer ID), InvoiceDate ở dạng chuỗi",
    "Đổi sang snake_case, chuyển invoice_date sang datetime và cắt khoảng trắng thừa",
    len(df), "Đảm bảo tính nhất quán khi trích xuất đặc trưng thời gian và ghép bảng"
)

# 4.2 Loại bỏ bản ghi trùng lặp hoàn toàn (Exact Duplicates)
n_dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log_step(
    "4.2", "all",
    f"{n_dups:,} dòng trùng lặp hoàn toàn trên tất cả các cột",
    "Xóa các dòng trùng lặp, giữ lại bản ghi xuất hiện đầu tiên",
    n_dups, "Lỗi ghi nhận lặp từ hệ thống xuất dữ liệu giao dịch"
)
print(f"Sau khi xóa trùng lặp: {len(df):,} dòng")

Sau khi xóa trùng lặp: 1,033,036 dòng


In [5]:
# 3.1 Loại bỏ giao dịch thiếu customer_id (cơ chế MNAR/MAR - khách không đăng nhập)
n_missing_cid = df["customer_id"].isna().sum()
df = df.dropna(subset=["customer_id"]).copy()
df["customer_id"] = df["customer_id"].astype(int).astype(str)
log_step(
    "3.1", "customer_id",
    f"{n_missing_cid:,} dòng thiếu customer_id",
    "Loại bỏ các dòng không có customer_id và chuẩn hóa thành chuỗi mã số nguyên",
    n_missing_cid, "Đơn vị phân tích là hành vi từng khách hàng; không thể gán giao dịch vô danh"
)

# 3.2 Xử lý missing thật & missing ngụy trang ở description
disguised_desc = ["nan", "None", "", "?", "??", "check", "damaged"]
mask_desc = df["description"].isin(disguised_desc)
n_missing_desc = mask_desc.sum()
df.loc[mask_desc, "description"] = "UNKNOWN_PRODUCT"
log_step(
    "3.2", "description",
    f"{n_missing_desc:,} dòng thiếu hoặc chứa token lỗi ('?', 'damaged', 'check'...)",
    "Chuẩn hóa về nhãn 'UNKNOWN_PRODUCT' thay vì xóa dòng",
    n_missing_desc, "Bảo toàn lịch sử chi tiêu và tần suất mua của khách hàng"
)

# 3.3 Xử lý missing ngụy trang 'Unspecified' ở country (nếu còn sau bước 3.1)
n_unspec_country = (df["country"] == "Unspecified").sum()
if n_unspec_country > 0:
    # Kiểm tra xem customer_id đó có quốc gia ở đơn khác không để điền theo nhóm (MAR)
    cid_country_map = (
        df[df["country"] != "Unspecified"]
        .groupby("customer_id")["country"]
        .agg(lambda s: s.mode().iloc[0])
    )
    df["country"] = df.apply(
        lambda r: cid_country_map.get(r["customer_id"], "Unspecified")
        if r["country"] == "Unspecified" else r["country"],
        axis=1
    )
log_step(
    "3.3", "country",
    f"{n_unspec_country:,} dòng có giá trị 'Unspecified' sau khi lọc customer_id",
    "Suy diễn theo quốc gia phổ biến nhất của chính customer_id đó (MAR imputation), giữ nguyên nếu chưa xác định",
    n_unspec_country, "Tận dụng quan hệ khách hàng - quốc gia để tránh mất dữ liệu khi merge "
)

In [6]:
# 4.3 Loại bỏ mã phi sản phẩm (phí vận chuyển, chiết khấu, điều chỉnh công nợ)
non_product_codes = [
    "POST", "D", "M", "BANK CHARGES", "AMAZONFEE", "DOT",
    "CRUK", "S", "B", "ADJUST", "ADJUST2", "TEST001", "TEST002"
]
mask_non_prod = df["stockcode"].str.upper().isin(non_product_codes)
n_non_prod = mask_non_prod.sum()
df = df[~mask_non_prod].copy()
log_step(
    "4.3", "stockcode",
    f"{n_non_prod:,} dòng là phí vận chuyển/điều chỉnh kế toán ({non_product_codes[:4]}...)",
    "Loại bỏ khỏi bảng giao dịch hàng hóa",
    n_non_prod, "Không phản ánh hành vi lựa chọn sản phẩm của người mua"
)

# 4.4 Gắn cờ đơn hủy/trả hàng (Invoice bắt đầu bằng 'C')
df["is_cancelled"] = df["invoice"].str.startswith("C")
n_cancelled = df["is_cancelled"].sum()
log_step(
    "4.4", "invoice & quantity",
    f"{n_cancelled:,} dòng là đơn hủy/hoàn trả (Invoice bắt đầu bằng 'C', Quantity < 0)",
    "Giữ nguyên và tạo cột cờ is_cancelled=True để tính tỷ lệ hoàn đơn của từng khách",
    n_cancelled, "Sự kiện hành vi có thật (rare/behavioral event), rất hữu ích để dự đoán churn"
)

# 4.5 Loại bỏ giá trị bất khả thi (Price <= 0 hoặc dấu Quantity mâu thuẫn với trạng thái đơn)
mask_invalid = (
    (df["price"] <= 0)
    | ((~df["is_cancelled"]) & (df["quantity"] <= 0))
    | (df["is_cancelled"] & (df["quantity"] >= 0))
)
n_invalid = mask_invalid.sum()
df = df[~mask_invalid].copy()
log_step(
    "4.5", "price & quantity",
    f"{n_invalid:,} dòng có Price <= 0 hoặc dấu Quantity mâu thuẫn với loại hóa đơn",
    "Loại bỏ các bản ghi vi phạm quy tắc miền giá trị",
    n_invalid, "Hàng tặng 0 đồng hoặc lỗi nhập liệu gây sai lệch giá trị đơn hàng"
)

# 4.6 Chẩn đoán ngoại lai cực trị (IQR) trên tổng tiền dòng hàng
df["total_amount"] = df["quantity"] * df["price"]
q1, q3 = df["total_amount"].quantile([0.25, 0.75])
iqr = q3 - q1
n_outliers_iqr = ((df["total_amount"] < q1 - 1.5 * iqr) | (df["total_amount"] > q3 + 1.5 * iqr)).sum()
log_step(
    "4.6", "total_amount",
    f"{n_outliers_iqr:,} dòng nằm ngoài ngưỡng 1.5*IQR (khách mua buôn số lượng lớn)",
    "Không xóa; giữ lại để dùng biến đổi log1p và RobustScaler ",
    n_outliers_iqr, "Là khách hàng B2B giá trị cao có thật; xóa đi sẽ làm mất tín hiệu phân nhóm khách hàng"
)

In [9]:
print("=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH (RE-VALIDATE) ===")
display(audit(df))
display(check_disguised_missing(df))

log_df = pd.DataFrame(cleaning_log)
display(log_df)

log_df.to_csv("../reports/figures/cleaning_log.csv", index=False)
df.to_parquet("../data/processed/retail_cleaned_w4.parquet", index=False)
print(f"\nSố dòng sạch còn lại: {len(df):,} dòng.")

=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH (RE-VALIDATE) ===


,dtype,n_missing,pct_missing,n_unique,sample
invoice,str,0,0.0,43887,489434
stockcode,str,0,0.0,4636,85048
description,str,0,0.0,5246,15CM CHRISTMAS GLASS BALL 20 LIGHTS
quantity,int64,0,0.0,639,12
price,float64,0,0.0,477,6.95
customer_id,str,0,0.0,5876,13085
country,str,0,0.0,41,United Kingdom
invoice_date,datetime64[us],0,0.0,40630,2009-12-01 07:45:00
is_cancelled,bool,0,0.0,2,False
total_amount,float64,0,0.0,5002,83.4


,column,n_disguised,examples
0,country,519,[Unspecified]


,Step,Column,Finding,Action,Rows,Why
0,4.1,all_columns,"Tên cột có khoảng trắng (Customer ID), Invoice...","Đổi sang snake_case, chuyển invoice_date sang ...",1067371,Đảm bảo tính nhất quán khi trích xuất đặc trưn...
1,4.2,all,"34,335 dòng trùng lặp hoàn toàn trên tất cả cá...","Xóa các dòng trùng lặp, giữ lại bản ghi xuất h...",34335,Lỗi ghi nhận lặp từ hệ thống xuất dữ liệu giao...
2,3.1,customer_id,"235,151 dòng thiếu customer_id",Loại bỏ các dòng không có customer_id và chuẩn...,235151,Đơn vị phân tích là hành vi từng khách hàng; k...
3,3.2,description,"0 dòng thiếu hoặc chứa token lỗi ('?', 'damage...",Chuẩn hóa về nhãn 'UNKNOWN_PRODUCT' thay vì xó...,0,Bảo toàn lịch sử chi tiêu và tần suất mua của ...
4,3.3,country,521 dòng có giá trị 'Unspecified' sau khi lọc ...,Suy diễn theo quốc gia phổ biến nhất của chính...,521,Tận dụng quan hệ khách hàng - quốc gia để trán...
5,3.1,customer_id,0 dòng thiếu customer_id,Loại bỏ các dòng không có customer_id và chuẩn...,0,Đơn vị phân tích là hành vi từng khách hàng; k...
6,3.2,description,"0 dòng thiếu hoặc chứa token lỗi ('?', 'damage...",Chuẩn hóa về nhãn 'UNKNOWN_PRODUCT' thay vì xó...,0,Bảo toàn lịch sử chi tiêu và tần suất mua của ...
7,3.3,country,521 dòng có giá trị 'Unspecified' sau khi lọc ...,Suy diễn theo quốc gia phổ biến nhất của chính...,521,Tận dụng quan hệ khách hàng - quốc gia để trán...
8,4.3,stockcode,"3,387 dòng là phí vận chuyển/điều chỉnh kế toá...",Loại bỏ khỏi bảng giao dịch hàng hóa,3387,Không phản ánh hành vi lựa chọn sản phẩm của n...
9,4.4,invoice & quantity,"17,593 dòng là đơn hủy/hoàn trả (Invoice bắt đ...",Giữ nguyên và tạo cột cờ is_cancelled=True để ...,17593,Sự kiện hành vi có thật (rare/behavioral event...



Số dòng sạch còn lại: 794,437 dòng.
